# Oregon ACS → Flourish Data Preparation

## Purpose

Prepare the cleaned **2024 American Community Survey (ACS)** data for an interactive Flourish visualization focused on Oregon.

This visualization supports the transition from identifying **Oregon as a priority state** to examining the community conditions that may shape access to pediatric mental health care within the state.

### Question

**What community-level factors may shape pediatric mental health access within Oregon?**

### Data source

**American Community Survey (ACS), 2024**

Cleaned project dataset:

`ACS_2024_cleaned_state_district.csv`

The cleaned dataset contains:

- **10,953 rows**
- **176 variables**
- State and school-district-level ACS geography

### Visualization goal

Create a **hierarchical bubble visualization** in Flourish.

The first level will show broad community-factor categories as large bubbles. Selecting a category will reveal the individual ACS measures contained within it.

Potential factor groups include:

- Economic conditions
- Housing
- Insurance / healthcare access
- Transportation
- Digital access
- Household / family context

### Workflow

1. Load the cleaned ACS dataset.
2. Isolate Oregon geographies.
3. Identify the relevant ACS measures.
4. Organize measures into interpretable factor groups.
5. Create a simplified hierarchy dataset for Flourish.
6. Export the final visualization-ready CSV.

> **Note:** ACS variables will be verified against their source definitions before being assigned human-readable labels or factor groups.

In [3]:
from pathlib import Path
import pandas as pd

# Notebook location:
# pediatric-mental-health-analysis/notebooks/Flourish_Animations/

PROJECT_ROOT = Path.cwd().parent.parent

acs_path = (
    PROJECT_ROOT
    / "data"
    / "cleaned"
    / "ACS_2024_cleaned_state_district.csv"
)

print("Project root:", PROJECT_ROOT)
print("Loading:", acs_path)
print("File exists:", acs_path.exists())

acs = pd.read_csv(acs_path)

print("Dataset shape:", acs.shape)

Project root: c:\Users\akila\pediatric-mental-health-analysis
Loading: c:\Users\akila\pediatric-mental-health-analysis\data\cleaned\ACS_2024_cleaned_state_district.csv
File exists: True
Dataset shape: (10953, 176)


In [4]:
# ============================================================
# 1. INSPECT GEOGRAPHY STRUCTURE
# Question:
# How are state and school-district geographies identified
# in the cleaned ACS dataset?
# ============================================================

# Find likely geography / identifier columns
geo_cols = [
    col for col in acs.columns
    if any(
        term in col.upper()
        for term in ["NAME", "GEO", "STATE", "FIPS"]
    )
]

print("Potential geography columns:")
print(geo_cols)

# Display the geography fields for the first 20 rows
print("\nSample geography values:")
display(acs[geo_cols].head(20))

# Look specifically for rows containing Oregon
name_cols = [
    col for col in acs.columns
    if "NAME" in col.upper()
]

print("\nRows containing 'Oregon':")

for col in name_cols:
    matches = acs[
        acs[col].astype(str).str.contains(
            "Oregon",
            case=False,
            na=False
        )
    ]

    if len(matches) > 0:
        print(f"\nColumn: {col}")
        print(f"Matches: {len(matches)}")
        display(matches[geo_cols].head(20))

Potential geography columns:
['GEO_ID', 'NAME', 'FIPSST']

Sample geography values:


,GEO_ID,NAME,FIPSST
0,0400000US01,Alabama,1
1,0400000US02,Alaska,2
2,0400000US04,Arizona,4
3,0400000US05,Arkansas,5
4,0400000US06,California,6
5,0400000US08,Colorado,8
6,0400000US09,Connecticut,9
7,0400000US10,Delaware,10
8,0400000US11,District of Columbia,11
9,0400000US12,Florida,12



Rows containing 'Oregon':

Column: NAME
Matches: 194


,GEO_ID,NAME,FIPSST
37,0400000US41,Oregon,41
1866,9700000US1730160,"Oregon Community Unit School District 220, Ill...",17
2174,9700000US1808460,"Oregon-Davis School Corporation, Indiana",18
4783,9700000US2916860,"Oregon-Howell R-III School District, Missouri",29
6909,9700000US3904460,"Oregon City School District, Ohio",39
7827,9700000US4100003,"Falls City School District 57, Oregon",41
7828,9700000US4100014,"Vale School District 84, Oregon",41
7829,9700000US4100015,"Gervais School District 1, Oregon",41
7830,9700000US4100016,"Yamhill-Carlton School District 1, Oregon",41
7831,9700000US4100019,"Harrisburg School District 7J, Oregon",41


In [5]:
# ============================================================
# 2. ISOLATE OREGON SCHOOL-DISTRICT GEOGRAPHIES
# Question:
# Which ACS school-district geographies belong to Oregon?
# ============================================================

# Oregon state FIPS = 41
oregon = acs[acs["FIPSST"] == 41].copy()

print("All Oregon rows:", len(oregon))

# GEO_ID values beginning with 970 identify the school-district
# geography rows in this cleaned dataset.
oregon_districts = oregon[
    oregon["GEO_ID"].astype(str).str.startswith("970")
].copy()

print("Oregon district rows:", len(oregon_districts))

# Check the resulting geography
display(
    oregon_districts[
        ["GEO_ID", "NAME", "FIPSST"]
    ].head(20)
)

# Verify that we did not accidentally retain another state
print("\nState FIPS values retained:")
print(oregon_districts["FIPSST"].value_counts())

print("\nExample district names:")
print(
    oregon_districts["NAME"]
    .head(10)
    .to_string(index=False)
)

All Oregon rows: 189
Oregon district rows: 188


,GEO_ID,NAME,FIPSST
7827,9700000US4100003,"Falls City School District 57, Oregon",41
7828,9700000US4100014,"Vale School District 84, Oregon",41
7829,9700000US4100015,"Gervais School District 1, Oregon",41
7830,9700000US4100016,"Yamhill-Carlton School District 1, Oregon",41
7831,9700000US4100019,"Harrisburg School District 7J, Oregon",41
7832,9700000US4100020,"North Santiam School District 29J, Oregon",41
7833,9700000US4100021,"South Wasco County School District 1, Oregon",41
7834,9700000US4100023,"Hillsboro School District 1J, Oregon",41
7835,9700000US4100040,"Knappa School District 4, Oregon",41
7836,9700000US4100047,"Ione School District 2, Oregon",41



State FIPS values retained:
FIPSST
41    188
Name: count, dtype: int64

Example district names:
       Falls City School District 57, Oregon
             Vale School District 84, Oregon
           Gervais School District 1, Oregon
   Yamhill-Carlton School District 1, Oregon
       Harrisburg School District 7J, Oregon
   North Santiam School District 29J, Oregon
South Wasco County School District 1, Oregon
        Hillsboro School District 1J, Oregon
            Knappa School District 4, Oregon
              Ione School District 2, Oregon


In [6]:
# ============================================================
# 3. INSPECT AVAILABLE ACS MEASURES
# Question:
# Which ACS variables are available to describe community
# conditions across Oregon school-district geographies?
# ============================================================

# Exclude geography identifiers
measure_cols = [
    col for col in oregon_districts.columns
    if col not in ["GEO_ID", "NAME", "FIPSST"]
]

print("Number of ACS measure columns:", len(measure_cols))

# Group variables by ACS profile
profiles = ["DP02", "DP03", "DP04", "DP05"]

for profile in profiles:
    cols = [c for c in measure_cols if c.startswith(profile)]
    
    print(f"\n{'=' * 60}")
    print(f"{profile}: {len(cols)} variables")
    print("=" * 60)
    
    for col in cols:
        print(col)

# ------------------------------------------------------------
# Look specifically for percentage-estimate variables.
#
# ACS profile variables ending in PE are percentage estimates,
# which are generally more useful than raw counts when
# comparing districts of very different population sizes.
# ------------------------------------------------------------

percent_cols = [
    col for col in measure_cols
    if col.endswith("PE")
]

print("\n" + "=" * 60)
print("PERCENTAGE ESTIMATE VARIABLES")
print("=" * 60)

print("Count:", len(percent_cols))

for col in percent_cols:
    print(col)

Number of ACS measure columns: 173

DP02: 48 variables
DP02_0003E
DP02_0003PE
DP02_0003PM
DP02_0005E
DP02_0005PE
DP02_0005PM
DP02_0007E
DP02_0007PE
DP02_0007PM
DP02_0011E
DP02_0011PE
DP02_0011PM
DP02_0014E
DP02_0014PE
DP02_0014PM
DP02_0056E
DP02_0056PE
DP02_0056PM
DP02_0057E
DP02_0057PE
DP02_0057PM
DP02_0067E
DP02_0067PE
DP02_0067PM
DP02_0068E
DP02_0068PE
DP02_0068PM
DP02_0073E
DP02_0073PE
DP02_0073PM
DP02_0074E
DP02_0074PE
DP02_0074PM
DP02_0113E
DP02_0113PE
DP02_0113PM
DP02_0114E
DP02_0114PE
DP02_0114PM
DP02_0115E
DP02_0115PE
DP02_0115PM
DP02_0153E
DP02_0153PE
DP02_0153PM
DP02_0154E
DP02_0154PE
DP02_0154PM

DP03: 31 variables
DP03_0009E
DP03_0009M
DP03_0016E
DP03_0016PE
DP03_0016PM
DP03_0017E
DP03_0017PE
DP03_0017PM
DP03_0062E
DP03_0062M
DP03_0072E
DP03_0072PE
DP03_0072PM
DP03_0074E
DP03_0074PE
DP03_0074PM
DP03_0100E
DP03_0100PE
DP03_0100PM
DP03_0101E
DP03_0101PE
DP03_0101PM
DP03_0120E
DP03_0120PE
DP03_0120PM
DP03_0129E
DP03_0129PE
DP03_0129PM
DP03_0132E
DP03_0132PE
DP03_0132PM

DP04:

In [7]:
# ============================================================
# 4. IDENTIFY ANALYSIS-READY / DERIVED VARIABLES
# Question:
# Which cleaned ACS variables are already available in a
# human-readable or derived form for community analysis?
# ============================================================

# ACS source variables begin with DP02, DP03, DP04, or DP05.
# Anything else (excluding geography fields) may be a derived
# variable created during the cleaning process.

source_prefixes = ("DP02_", "DP03_", "DP04_", "DP05_")
geo_fields = {"GEO_ID", "NAME", "FIPSST"}

derived_cols = [
    col for col in oregon_districts.columns
    if col not in geo_fields
    and not col.startswith(source_prefixes)
]

print("Derived / analysis-ready columns:")
print(f"Count: {len(derived_cols)}")

for col in derived_cols:
    print(f" - {col}")

# Show values for Oregon districts if derived columns exist
if derived_cols:
    display(
        oregon_districts[
            ["NAME"] + derived_cols
        ].head(10)
    )

Derived / analysis-ready columns:
Count: 3
 - pct_has_vehicle
 - pct_complete_plumbing
 - pct_complete_kitchen


,NAME,pct_has_vehicle,pct_complete_plumbing,pct_complete_kitchen
7827,"Falls City School District 57, Oregon",90.3,98.3,98.3
7828,"Vale School District 84, Oregon",91.4,98.2,97.2
7829,"Gervais School District 1, Oregon",96.3,100.0,100.0
7830,"Yamhill-Carlton School District 1, Oregon",96.8,99.8,98.8
7831,"Harrisburg School District 7J, Oregon",100.0,100.0,100.0
7832,"North Santiam School District 29J, Oregon",95.7,99.5,97.8
7833,"South Wasco County School District 1, Oregon",97.3,99.2,97.6
7834,"Hillsboro School District 1J, Oregon",94.9,99.7,99.4
7835,"Knappa School District 4, Oregon",98.6,99.3,99.2
7836,"Ione School District 2, Oregon",100.0,100.0,100.0


In [8]:
# ============================================================
# 5. BUILD CANDIDATE ACS VARIABLE INVENTORY
# Question:
# Which retained ACS measures should be considered for the
# Oregon community-context visualization?
#
# Project documentation identifies these priority domains:
# - child poverty
# - uninsured youth
# - SNAP
# - female-headed households with children
# - limited English
# - rent burden
# - broadband
# - vehicle access
# ============================================================

# Focus on percentage estimates first
percent_cols = [
    c for c in acs.columns
    if c.startswith(("DP02_", "DP03_", "DP04_", "DP05_"))
    and c.endswith("PE")
]

print("ACS percentage-estimate columns:", len(percent_cols))

# Oregon statewide row
oregon_state = acs[
    (acs["FIPSST"] == 41) &
    (acs["NAME"] == "Oregon")
].copy()

print("Oregon statewide rows:", len(oregon_state))

# Create compact inventory:
# code + Oregon statewide value + district distribution
inventory = []

for col in percent_cols:
    
    district_values = pd.to_numeric(
        oregon_districts[col],
        errors="coerce"
    )
    
    state_value = pd.to_numeric(
        oregon_state[col],
        errors="coerce"
    )
    
    inventory.append({
        "variable": col,
        "oregon_state_pct":
            state_value.iloc[0] if len(state_value) else None,
        "district_nonmissing": district_values.notna().sum(),
        "district_mean": district_values.mean(),
        "district_min": district_values.min(),
        "district_max": district_values.max()
    })

inventory_df = pd.DataFrame(inventory)

# Remove variables with no usable Oregon district data
inventory_df = inventory_df[
    inventory_df["district_nonmissing"] > 0
].copy()

# Sort by ACS profile and variable code
inventory_df = inventory_df.sort_values("variable")

print("\nUsable percentage measures:", len(inventory_df))

display(inventory_df)

ACS percentage-estimate columns: 54
Oregon statewide rows: 1

Usable percentage measures: 54


,variable,oregon_state_pct,district_nonmissing,district_mean,district_min,district_max
0,DP02_0003PE,16.9,188,16.495745,0.0,46.8
1,DP02_0005PE,2.5,188,2.400532,0.0,20.8
2,DP02_0007PE,1.4,188,1.140957,0.0,5.4
3,DP02_0011PE,4.0,188,3.139894,0.0,10.5
4,DP02_0014PE,27.2,188,26.227660,0.0,48.0
5,DP02_0056PE,41.2,186,46.929032,13.0,82.3
6,DP02_0057PE,22.1,186,24.469355,0.0,52.7
7,DP02_0067PE,91.8,188,91.056383,69.3,100.0
8,DP02_0068PE,36.8,188,25.161702,7.3,81.5
9,DP02_0073PE,845822.0,188,4499.053191,0.0,78296.0


In [9]:
# ============================================================
# 6. LOCATE ACS VARIABLE DEFINITIONS
# Question:
# What do the retained DP02, DP03, DP04, and DP05 variable
# codes actually represent?
#
# We will use the original Census metadata rather than infer
# variable meanings from their values.
# ============================================================

from pathlib import Path

acs_raw_root = PROJECT_ROOT / "ACS" / "Extracted"

print("ACS source directory:", acs_raw_root)
print("Directory exists:", acs_raw_root.exists())

# Find metadata / label files inside each extracted ACS profile
metadata_files = []

for path in acs_raw_root.rglob("*"):
    if path.is_file() and path.suffix.lower() in {
        ".csv", ".txt", ".json"
    }:
        metadata_files.append(path)

print(f"\nCandidate source/metadata files: {len(metadata_files)}")

for path in metadata_files:
    print(path.relative_to(PROJECT_ROOT))

ACS source directory: c:\Users\akila\pediatric-mental-health-analysis\ACS\Extracted
Directory exists: True

Candidate source/metadata files: 12
ACS\Extracted\ACSDP5Y2024.DP02_2026-08-08T192321\ACSDP5Y2024.DP02-Column-Metadata.csv
ACS\Extracted\ACSDP5Y2024.DP02_2026-08-08T192321\ACSDP5Y2024.DP02-Data.csv
ACS\Extracted\ACSDP5Y2024.DP02_2026-08-08T192321\ACSDP5Y2024.DP02-Table-Notes.txt
ACS\Extracted\ACSDP5Y2024.DP03_2026-08-08T192255\ACSDP5Y2024.DP03-Column-Metadata.csv
ACS\Extracted\ACSDP5Y2024.DP03_2026-08-08T192255\ACSDP5Y2024.DP03-Data.csv
ACS\Extracted\ACSDP5Y2024.DP03_2026-08-08T192255\ACSDP5Y2024.DP03-Table-Notes.txt
ACS\Extracted\ACSDP5Y2024.DP04_2026-08-08T192236\ACSDP5Y2024.DP04-Column-Metadata.csv
ACS\Extracted\ACSDP5Y2024.DP04_2026-08-08T192236\ACSDP5Y2024.DP04-Data.csv
ACS\Extracted\ACSDP5Y2024.DP04_2026-08-08T192236\ACSDP5Y2024.DP04-Table-Notes.txt
ACS\Extracted\ACSDP5Y2024.DP05_2026-08-08T191722\ACSDP5Y2024.DP05-Column-Metadata.csv
ACS\Extracted\ACSDP5Y2024.DP05_2026-08-08

In [10]:
# ============================================================
# 7. DECODE RETAINED ACS VARIABLES
# Question:
# What do the 54 retained ACS percentage measures represent?
# ============================================================

metadata_paths = list(
    acs_raw_root.rglob("*Column-Metadata.csv")
)

print("Metadata files found:", len(metadata_paths))

metadata_frames = []

for path in metadata_paths:
    meta = pd.read_csv(path)

    print(f"\n{path.name}")
    print("Columns:", meta.columns.tolist())

    # Standard Census metadata files use:
    # Column Name | Label
    if "Column Name" in meta.columns and "Label" in meta.columns:
        temp = meta[["Column Name", "Label"]].copy()
        temp.columns = ["variable", "label"]
        metadata_frames.append(temp)

# Combine all four ACS profiles
acs_labels = pd.concat(
    metadata_frames,
    ignore_index=True
).drop_duplicates(subset="variable")

print("\nDecoded metadata rows:", len(acs_labels))

# Join definitions to the 54 usable percentage measures
decoded_inventory = inventory_df.merge(
    acs_labels,
    on="variable",
    how="left"
)

# Put the human-readable definition next to the code
decoded_inventory = decoded_inventory[
    [
        "variable",
        "label",
        "oregon_state_pct",
        "district_mean",
        "district_min",
        "district_max",
        "district_nonmissing"
    ]
]

print(
    "\nRetained Oregon ACS percentage measures "
    "with Census definitions:"
)

display(decoded_inventory)

Metadata files found: 4

ACSDP5Y2024.DP02-Column-Metadata.csv
Columns: ['Column Name', 'Label']

ACSDP5Y2024.DP03-Column-Metadata.csv
Columns: ['Column Name', 'Label']

ACSDP5Y2024.DP04-Column-Metadata.csv
Columns: ['Column Name', 'Label']

ACSDP5Y2024.DP05-Column-Metadata.csv
Columns: ['Column Name', 'Label']

Decoded metadata rows: 2170

Retained Oregon ACS percentage measures with Census definitions:


,variable,label,oregon_state_pct,district_mean,district_min,district_max,district_nonmissing
0,DP02_0003PE,Percent!!HOUSEHOLDS BY TYPE!!Total households!...,16.9,16.495745,0.0,46.8,188
1,DP02_0005PE,Percent!!HOUSEHOLDS BY TYPE!!Total households!...,2.5,2.400532,0.0,20.8,188
2,DP02_0007PE,Percent!!HOUSEHOLDS BY TYPE!!Total households!...,1.4,1.140957,0.0,5.4,188
3,DP02_0011PE,Percent!!HOUSEHOLDS BY TYPE!!Total households!...,4.0,3.139894,0.0,10.5,188
4,DP02_0014PE,Percent!!HOUSEHOLDS BY TYPE!!Total households!...,27.2,26.227660,0.0,48.0,188
5,DP02_0056PE,Percent!!SCHOOL ENROLLMENT!!Population 3 years...,41.2,46.929032,13.0,82.3,186
6,DP02_0057PE,Percent!!SCHOOL ENROLLMENT!!Population 3 years...,22.1,24.469355,0.0,52.7,186
7,DP02_0067PE,Percent!!EDUCATIONAL ATTAINMENT!!Population 25...,91.8,91.056383,69.3,100.0,188
8,DP02_0068PE,Percent!!EDUCATIONAL ATTAINMENT!!Population 25...,36.8,25.161702,7.3,81.5,188
9,DP02_0073PE,Percent!!DISABILITY STATUS OF THE CIVILIAN NON...,845822.0,4499.053191,0.0,78296.0,188


In [11]:
# ============================================================
# 8. FIND COMMUNITY-FACTOR MEASURES
# Question:
# Which retained ACS measures describe community conditions
# that may be relevant to pediatric mental health access?
# ============================================================

# Keywords tied to the community-context domains we care about
keywords = [
    "poverty",
    "health insurance",
    "food stamps",
    "snap",
    "children",
    "female householder",
    "language",
    "english",
    "internet",
    "broadband",
    "vehicle",
    "rent",
    "employment",
    "income"
]

pattern = "|".join(keywords)

candidate_factors = decoded_inventory[
    decoded_inventory["label"]
    .str.contains(pattern, case=False, na=False)
].copy()

# Shorten Census's hierarchical label formatting for inspection
candidate_factors["clean_label"] = (
    candidate_factors["label"]
    .str.replace("Percent!!", "", regex=False)
    .str.replace("Estimate!!", "", regex=False)
    .str.replace("!!", " → ", regex=False)
)

candidate_factors = candidate_factors[
    [
        "variable",
        "clean_label",
        "oregon_state_pct",
        "district_mean",
        "district_min",
        "district_max",
        "district_nonmissing"
    ]
]

print("Candidate community-factor measures:", len(candidate_factors))

pd.set_option("display.max_colwidth", None)

display(candidate_factors)

Candidate community-factor measures: 27


,variable,clean_label,oregon_state_pct,district_mean,district_min,district_max,district_nonmissing
0,DP02_0003PE,HOUSEHOLDS BY TYPE → Total households → Married-couple household → With children of the householder under 18 years,16.9,16.495745,0.0,46.8,188
1,DP02_0005PE,HOUSEHOLDS BY TYPE → Total households → Cohabiting couple household → With children of the householder under 18 years,2.5,2.400532,0.0,20.8,188
2,DP02_0007PE,"HOUSEHOLDS BY TYPE → Total households → Male householder, no spouse/partner present → With children of the householder under 18 years",1.4,1.140957,0.0,5.4,188
3,DP02_0011PE,"HOUSEHOLDS BY TYPE → Total households → Female householder, no spouse/partner present → With children of the householder under 18 years",4.0,3.139894,0.0,10.5,188
11,DP02_0113PE,LANGUAGE SPOKEN AT HOME → Population 5 years and over → English only,84.8,91.056915,42.8,100.0,188
12,DP02_0114PE,LANGUAGE SPOKEN AT HOME → Population 5 years and over → Language other than English,15.2,8.943085,0.0,57.2,188
13,DP02_0115PE,"LANGUAGE SPOKEN AT HOME → Population 5 years and over → Language other than English → Speak English less than very well""""",5.5,3.219681,0.0,27.9,188
14,DP02_0153PE,COMPUTERS AND INTERNET USE → Total households → With a computer,96.6,94.337234,55.1,100.0,188
15,DP02_0154PE,COMPUTERS AND INTERNET USE → Total households → With a broadband Internet subscription,92.4,88.528191,38.1,99.6,188
16,DP03_0016PE,EMPLOYMENT STATUS → Own children of the householder 6 to 17 years,562429.0,2991.643617,0.0,52547.0,188


In [12]:
# ============================================================
# 9. SELECT FINAL COMMUNITY FACTORS
# Question:
# Across Oregon school-district geographies, how widespread
# are community conditions that may shape access to pediatric
# mental health care?
#
# Bubble logic:
# BIG bubble    = community factor category
# SMALL bubble  = individual ACS measure
# Bubble value  = % of Oregon districts above the Oregon
#                 statewide benchmark for that measure
# ============================================================

factor_map = {
    
    # ECONOMIC PRESSURE
    "DP03_0074PE": {
        "category": "Economic pressure",
        "factor": "Households receiving SNAP"
    },
    "DP03_0129PE": {
        "category": "Economic pressure",
        "factor": "Children under 18 in poverty"
    },
    "DP03_0132PE": {
        "category": "Economic pressure",
        "factor": "Children ages 5–17 in poverty"
    },

    # FAMILY CONTEXT
    "DP02_0011PE": {
        "category": "Family context",
        "factor": "Female householder with children"
    },
    "DP03_0017PE": {
        "category": "Family context",
        "factor": "All parents in labor force"
    },

    # ACCESS BARRIERS
    "DP03_0101PE": {
        "category": "Access barriers",
        "factor": "Children under 19 uninsured"
    },
    "DP02_0115PE": {
        "category": "Access barriers",
        "factor": "Limited English proficiency"
    },

    # For broadband, the ACS variable is access.
    # We convert it to NO broadband below.
    "DP02_0154PE": {
        "category": "Access barriers",
        "factor": "No broadband subscription",
        "reverse": True
    },

    # HOUSING & TRANSPORTATION
    "DP04_0058PE": {
        "category": "Housing & transportation",
        "factor": "No vehicle available"
    },
    "DP04_0115PE": {
        "category": "Housing & transportation",
        "factor": "High homeowner cost burden"
    },
    "DP04_0142PE": {
        "category": "Housing & transportation",
        "factor": "High rent burden"
    }
}


# ------------------------------------------------------------
# Calculate Oregon district prevalence relative to the
# statewide ACS benchmark.
# ------------------------------------------------------------

flourish_rows = []

for variable, info in factor_map.items():

    district_values = pd.to_numeric(
        oregon_districts[variable],
        errors="coerce"
    )

    state_value = pd.to_numeric(
        oregon_state[variable],
        errors="coerce"
    ).iloc[0]

    # Convert positive-access measure into barrier measure
    if info.get("reverse", False):
        district_values = 100 - district_values
        state_value = 100 - state_value

    valid = district_values.dropna()

    districts_above = (valid > state_value).sum()
    pct_districts_above = (
        districts_above / len(valid) * 100
        if len(valid) > 0
        else None
    )

    flourish_rows.append({
        "category": info["category"],
        "factor": info["factor"],
        "variable": variable,
        "oregon_state_value": round(state_value, 1),
        "districts_with_data": len(valid),
        "districts_above_state": districts_above,
        "pct_districts_above_state": round(
            pct_districts_above, 1
        )
    })


flourish_factors = pd.DataFrame(flourish_rows)

display(flourish_factors)

,category,factor,variable,oregon_state_value,districts_with_data,districts_above_state,pct_districts_above_state
0,Economic pressure,Households receiving SNAP,DP03_0074PE,16.0,188,96,51.1
1,Economic pressure,Children under 18 in poverty,DP03_0129PE,13.2,186,96,51.6
2,Economic pressure,Children ages 5–17 in poverty,DP03_0132PE,12.1,186,88,47.3
3,Family context,Female householder with children,DP02_0011PE,4.0,188,56,29.8
4,Family context,All parents in labor force,DP03_0017PE,71.8,186,77,41.4
5,Access barriers,Children under 19 uninsured,DP03_0101PE,3.0,186,92,49.5
6,Access barriers,Limited English proficiency,DP02_0115PE,5.5,188,32,17.0
7,Access barriers,No broadband subscription,DP02_0154PE,7.6,188,135,71.8
8,Housing & transportation,No vehicle available,DP04_0058PE,7.1,188,28,14.9
9,Housing & transportation,High homeowner cost burden,DP04_0115PE,22.9,188,108,57.4


In [13]:
# ============================================================
# 10. CREATE FLOURISH HIERARCHY-BUBBLE DATA
#
# Visualization structure:
#
# Oregon
#   ├── Economic pressure
#   │     ├── Households receiving SNAP
#   │     ├── Children under 18 in poverty
#   │     └── Children ages 5–17 in poverty
#   │
#   ├── Family context
#   │     ├── Female householder with children
#   │     └── All parents in labor force
#   │
#   ├── Access barriers
#   │     ├── Children under 19 uninsured
#   │     ├── Limited English proficiency
#   │     └── No broadband subscription
#   │
#   └── Housing & transportation
#         ├── No vehicle available
#         ├── High homeowner cost burden
#         └── High rent burden
#
# Bubble size:
# % of Oregon school-district geographies above the
# Oregon statewide benchmark.
# ============================================================

# Create leaf-level rows
flourish_export = flourish_factors.copy()

flourish_export["state"] = "Oregon"

flourish_export = flourish_export[
    [
        "state",
        "category",
        "factor",
        "pct_districts_above_state",
        "oregon_state_value",
        "districts_above_state",
        "districts_with_data"
    ]
].copy()

# Rename for easy Flourish setup
flourish_export.columns = [
    "State",
    "Category",
    "Factor",
    "Value",
    "Oregon benchmark",
    "Districts above benchmark",
    "Districts with data"
]

# Sort categories/factors for clean upload
category_order = [
    "Economic pressure",
    "Family context",
    "Access barriers",
    "Housing & transportation"
]

flourish_export["Category"] = pd.Categorical(
    flourish_export["Category"],
    categories=category_order,
    ordered=True
)

flourish_export = (
    flourish_export
    .sort_values(
        ["Category", "Value"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

# Convert category back to plain text before CSV export
flourish_export["Category"] = (
    flourish_export["Category"].astype(str)
)

display(flourish_export)


# ------------------------------------------------------------
# EXPORT
# ------------------------------------------------------------

export_dir = PROJECT_ROOT / "data" / "flourish_export"
export_dir.mkdir(parents=True, exist_ok=True)

output_path = (
    export_dir /
    "oregon_acs_hierarchy_bubbles.csv"
)

flourish_export.to_csv(
    output_path,
    index=False
)

print("\nFlourish file saved:")
print(output_path)

,State,Category,Factor,Value,Oregon benchmark,Districts above benchmark,Districts with data
0,Oregon,Economic pressure,Children under 18 in poverty,51.6,13.2,96,186
1,Oregon,Economic pressure,Households receiving SNAP,51.1,16.0,96,188
2,Oregon,Economic pressure,Children ages 5–17 in poverty,47.3,12.1,88,186
3,Oregon,Family context,All parents in labor force,41.4,71.8,77,186
4,Oregon,Family context,Female householder with children,29.8,4.0,56,188
5,Oregon,Access barriers,No broadband subscription,71.8,7.6,135,188
6,Oregon,Access barriers,Children under 19 uninsured,49.5,3.0,92,186
7,Oregon,Access barriers,Limited English proficiency,17.0,5.5,32,188
8,Oregon,Housing & transportation,High homeowner cost burden,57.4,22.9,108,188
9,Oregon,Housing & transportation,High rent burden,39.4,43.1,74,188



Flourish file saved:
c:\Users\akila\pediatric-mental-health-analysis\data\flourish_export\oregon_acs_hierarchy_bubbles.csv
